# MVP — Mesa de Servicio Inteligente (datos reales IX, anonimizados)

**Qué hace este notebook, en orden:**
1. Conecta Google Drive y localiza la base `incidentes_anonimizado.db`.
2. Carga la tabla `issues` (Jira) y calcula **sentimiento / frustración** sobre el `Resumen`.
3. Construye la **tendencia mensual** con la fecha `Creada`.
4. Arma los bloques de **Jira**, **Causas BTI** y **Operacional**.
5. Genera `resultados.json` y lo **publica directo al dashboard** (GitHub).

> La **descarga** de datos se hace desde el propio tablero (botones "Descargar"), no desde aquí.
> Requiere la base regenerada con el `anonimizar_jira.py` corregido (enmascara `Persona asignada`, `Descripción`, `Comentario` y cédulas). Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Conectar Google Drive y localizar la base
Monta tu Drive y busca el archivo `.db`. Si hay varias copias, usa la **más grande** (la real, ~80 MB). Si no la encuentra en Drive, la descarga por su ID como respaldo.

In [ ]:
from google.colab import drive       # utilidades de Colab para montar Drive
drive.mount('/content/drive')        # pide permiso y monta tu Drive en /content/drive

import glob, os                      # glob = buscar archivos por patron; os = tamaños/rutas
FILE_ID = '1FZhxpcYeKsriLIHpKSJlfZNjygXZXqmX'   # id del .db en Drive (respaldo por si no se encuentra)

# Busca TODAS las copias del archivo dentro de tu Drive (en cualquier subcarpeta)
cands = glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True)
for c in cands:
    print(round(os.path.getsize(c)/1e6, 1), 'MB ->', c)   # muestra tamaño y ruta de cada copia

if cands:
    RUTA_DB = max(cands, key=os.path.getsize)   # elige la copia mas grande = la verdadera
    print('USANDO:', RUTA_DB)
else:
    # Respaldo: si no aparece en Drive, la baja por su ID con gdown
    import gdown
    RUTA_DB = '/content/incidentes_anonimizado.db'
    gdown.download(id=FILE_ID, output=RUTA_DB, quiet=False)

## 2. Cargar la tabla `issues` (Jira)
Abre la base SQLite, lista sus tablas y carga `issues` en un DataFrame de pandas. La función `load()` normaliza los nombres de columna a minúsculas para no depender de mayúsculas/espacios.

In [ ]:
import sqlite3, pandas as pd, numpy as np, re, json   # sqlite3=BD, pandas=tablas, re=texto, json=salida
from datetime import datetime

con = sqlite3.connect(RUTA_DB)        # abre conexion a la base
# Lista los nombres de las tablas que existen en la base
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)['name'].tolist()
print('Tablas:', tablas)

def load(nombre, cols='*'):
    """Carga una tabla en un DataFrame; columnas en minuscula y sin espacios. Devuelve None si no existe."""
    if nombre in tablas:
        t = pd.read_sql('SELECT %s FROM "%s"' % (cols, nombre), con)
        t.columns = t.columns.str.lower().str.strip()
        return t
    return None

df = load('issues')                   # tabla principal: incidentes/issues de Jira
print('issues:', df.shape)            # (numero de filas, numero de columnas)
print('Columnas:', df.columns.tolist())
df.head(3)                            # vista previa de las 3 primeras filas

## 3. Sentimiento y nivel de frustración
Sobre el campo `Resumen` calculamos una **métrica de sentimiento** con un léxico simple (palabras positivas vs. negativas) y la convertimos en un **índice de frustración 0–100**. Luego clasificamos cada ticket en banda **bajo / medio / alto**.

*Nota:* es un método sencillo para el MVP. Para producción se usaría un modelo de sentimiento en español (p. ej. `pysentimiento`).

In [ ]:
COL_TEXTO = 'resumen' if 'resumen' in df.columns else df.select_dtypes('object').columns[0]  # columna de texto
df = df.drop_duplicates()                                   # quita filas repetidas
df[COL_TEXTO] = df[COL_TEXTO].astype(str).str.strip()       # asegura texto y quita espacios

# Listas de palabras (lexico). Amplialas si quieres afinar el sentimiento.
POS = set('gracias ok correcto solucionado resuelto disponible exitoso'.split())
NEG = set('error falla fallo sin no cancelacion cancelación pendiente caido caída bloqueo rechazo duplicado incidencia problema demora retraso urgente'.split())

def sent(t):
    """Devuelve un puntaje en [-1, 1]: (positivas - negativas) / (positivas + negativas)."""
    pal = re.findall(r'[a-záéíóúñ]+', str(t).lower())       # separa el texto en palabras
    p = sum(w in POS for w in pal)                          # cuenta palabras positivas
    n = sum(w in NEG for w in pal)                          # cuenta palabras negativas
    return round((p - n) / (p + n), 3) if (p + n) else 0.0  # 0 si no hay palabras de sentimiento

df['sentimiento'] = df[COL_TEXTO].apply(sent)               # aplica a cada fila
df['frustracion'] = ((1 - df['sentimiento']) / 2 * 100).round(1)   # convierte [-1,1] -> [0,100]
df['banda'] = df['frustracion'].apply(lambda f: 'alto' if f>=66 else ('medio' if f>=40 else 'bajo'))
df[['sentimiento','frustracion']].describe()                # resumen estadistico

## 4. Tendencia mensual
Usa la fecha de creación (`Creada`) para contar cuántos tickets entran por mes. Esto alimenta la gráfica de línea del tablero.

In [ ]:
# Busca la primera columna de fecha que exista
COL_FECHA = next((c for c in ['creada','fecha_creacion','fecha','created','actualizada'] if c in df.columns), None)
serie_mensual = {}
if COL_FECHA:
    fechas = pd.to_datetime(df[COL_FECHA], errors='coerce', utc=True)   # convierte texto a fecha (NaT si falla)
    serie = fechas.dt.to_period('M').astype(str).value_counts()         # agrupa por mes (AAAA-MM) y cuenta
    serie = serie[serie.index != 'NaT'].sort_index()                    # quita fechas invalidas y ordena
    serie_mensual = {str(k): int(v) for k, v in serie.items()}          # {'2026-01': 120, ...}
print('Tendencia mensual:', serie_mensual)

## 5. Construir todos los bloques y `resultados.json`
Arma cada bloque que consume el tablero:
- **KPIs** (totales, % alto riesgo, promedios).
- **Clasificación** (por Prioridad) y **distribución de riesgo**.
- **Sentimiento** (histograma) e **impulsores** del riesgo.
- **Jira** (total, por estado, recientes).
- **Causas BTI** (desde `ingresadas`).
- **Operacional** (órdenes).
- **Alertas** de alto riesgo.
Todo se guarda en `resultados.json`.

In [ ]:
# --- Clasificacion para el resumen: usa Prioridad; si no, Tipo o Estado ---
COL_LABEL = next((c for c in ['prioridad','tipo de incidencia','estado'] if c in df.columns), None)
top_cat = df[COL_LABEL].value_counts().head(8).to_dict() if COL_LABEL else {}

# --- Histograma de sentimiento (5 tramos) ---
bins=[-1,-0.6,-0.2,0.2,0.6,1.0001]; etq=['muy negativo','negativo','neutro','positivo','muy positivo']
hist=pd.cut(df['sentimiento'],bins=bins,labels=etq,include_lowest=True).value_counts().reindex(etq).fillna(0).astype(int)

# --- Impulsores: que categoria predomina entre los tickets de ALTA frustracion ---
alta=df[df['banda']=='alto']
impulsores=([{'variable':str(k)[:40],'importancia':round(float(v),3)} for k,v in alta[COL_LABEL].value_counts(normalize=True).head(6).items()] if (COL_LABEL and len(alta)) else [])

# --- Bloque Jira (desde issues) ---
idc = 'clave' if 'clave' in df.columns else df.columns[0]   # id del ticket (ITHD-xxxx)
est = 'estado' if 'estado' in df.columns else None          # estado del ticket
jira = {'total': int(len(df))}                              # total de issues
if est:
    jira['por_estado'] = {str(k):int(v) for k,v in df[est].value_counts().head(8).items()}   # conteo por estado
jira['recientes'] = [{'id':str(r[idc])[:18],'resumen':str(r[COL_TEXTO])[:80],'estado':(str(r[est]) if est else '')} for _,r in df.head(15).iterrows()]

# --- Alertas de alto riesgo (los 15 con mas frustracion) ---
def recom(f):
    """Recomendacion segun el nivel de frustracion."""
    return 'Contacto inmediato y plan de retencion' if f>=66 else ('Seguimiento proactivo' if f>=40 else 'Monitoreo estandar')
tickets=[{'id':str(r[idc])[:18],'cliente':str(r[idc])[:18],'clasificacion':(str(r[COL_LABEL])[:30] if COL_LABEL else ''),'frustracion':float(r['frustracion']),'riesgo':float(r['frustracion']),'recomendacion':recom(r['frustracion'])} for _,r in df.sort_values('frustracion',ascending=False).head(15).iterrows()]

# --- Bloque Causas BTI y Operacional (desde la tabla 'ingresadas') ---
bti_tabla=[]; operacional={}
ing = load('ingresadas')
if ing is not None:
    code = next((c for c in ['bti_clasificacion','clasificacion','bti'] if c in ing.columns), None)   # codigo BTI
    desc = next((c for c in ['resumen_problema','descripcion','detalle'] if c in ing.columns), None)  # descripcion del problema
    if code:
        conteo = ing[code].value_counts().head(15)          # top 15 causas BTI
        dmap = {}
        if desc:
            for _,row in ing.iterrows():                    # toma una descripcion por codigo
                k=str(row[code]); dmap.setdefault(k, str(row[desc]))
        bti_tabla=[{'codigo':str(k),'descripcion':dmap.get(str(k),'')[:90],'casos':int(v)} for k,v in conteo.items()]
    aco = next((c for c in ['accion_ops','accion','estado'] if c in ing.columns), None)   # accion operativa
    operacional['ingresadas_total']=int(len(ing))           # total de ORs ingresadas
    if aco:
        operacional['por_accion']={str(k):int(v) for k,v in ing[aco].value_counts().head(8).items()}
ordn = load('ordenes_diarias')
if ordn is not None:
    operacional['ordenes_total']=int(len(ordn))             # total de ordenes diarias

# --- Empaqueta TODO en el diccionario que consume el tablero ---
resultados={
 'generado': datetime.now().strftime('%Y-%m-%d %H:%M')+' (datos reales IX anonimizados)',
 'kpis':{'total_tickets':int(len(df)),'phishing_aislados':0,'procesados':int(len(df)),'pct_alto_riesgo':round(float((df['banda']=='alto').mean()*100),1),'frustracion_promedio':round(float(df['frustracion'].mean()),1),'sentimiento_promedio':round(float(df['sentimiento'].mean()),3)},
 'clasificacion':{str(k)[:30]:int(v) for k,v in top_cat.items()},
 'distribucion_riesgo':{k:int(v) for k,v in df['banda'].value_counts().reindex(['bajo','medio','alto']).fillna(0).astype(int).items()},
 'sentimiento_hist':{'bins':etq,'conteo':hist.tolist()},
 'impulsores':impulsores,'serie_mensual':serie_mensual,
 'jira':jira,'bti_tabla':bti_tabla,'operacional':operacional,'tickets_alto_riesgo':tickets}

# Guarda el archivo que leera el dashboard
with open('resultados.json','w',encoding='utf-8') as f:
    json.dump(resultados,f,ensure_ascii=False,indent=2)
print('KPIs:', resultados['kpis'])
print('Jira:', jira.get('total'),'| Causas BTI:',len(bti_tabla),'| Operacional:',operacional)
print('resultados.json generado correctamente.')

## 6. Publicar al dashboard (GitHub API)
Sube `resultados.json` al repo (`docs/data/`) y el dashboard de GitHub Pages se actualiza en ~1 minuto. **No** descarga nada a tu PC: las descargas se hacen desde el propio tablero.

**Token (una sola vez):** GitHub → Settings → Developer settings → Fine-grained tokens → repo `mesa-servicio-ia`, permiso **Contents: Read and write**. Se pide con `getpass`, así que no queda guardado en el notebook.

In [ ]:
import base64, requests, getpass

# Datos del repositorio destino
GH_OWNER='ricardocasallas3-ux'; GH_REPO='mesa-servicio-ia'
GH_PATH='docs/data/resultados.json'; GH_BRANCH='main'

token = getpass.getpass('Pega tu token de GitHub (no se mostrara): ')   # entrada segura del token
headers={'Authorization':'token '+token,'Accept':'application/vnd.github+json'}
url='https://api.github.com/repos/%s/%s/contents/%s' % (GH_OWNER,GH_REPO,GH_PATH)

# 1) Consulta si el archivo ya existe para obtener su 'sha' (necesario para reemplazarlo)
r=requests.get(url,headers=headers,params={'ref':GH_BRANCH})
sha=r.json().get('sha') if r.status_code==200 else None

# 2) Codifica el contenido en base64 (requisito de la API de GitHub)
with open('resultados.json','rb') as f:
    contenido=base64.b64encode(f.read()).decode()

# 3) Envia el archivo (crea o actualiza)
payload={'message':'Actualizar dashboard desde Colab','content':contenido,'branch':GH_BRANCH}
if sha: payload['sha']=sha
resp=requests.put(url,headers=headers,json=payload)

# 4) Resultado: 200/201 = OK; muestra el enlace del tablero
if resp.status_code in (200,201):
    print('OK. Mira el tablero en ~1 min: https://%s.github.io/%s/' % (GH_OWNER,GH_REPO))
else:
    print('Error %s:' % resp.status_code, resp.json())